# 🥇 Couche Gold : Modélisation Décisionnelle & Chargement dans PostgreSQL

---

## 🎯 Objectifs de ce Notebook
1. Comprendre la transition du **Data Lake** (MinIO Parquet) vers le **Data Warehouse** relationnel (PostgreSQL).
2. Modéliser des **Data Marts** analytiques selon les besoins du management.
3. Calculer 3 tables de faits agrégées avec PySpark :
   - Métriques quotidiennes par arrondissement (`agg_daily_taxi_metrics`)
   - Analyse de la congestion horaire (`agg_hourly_demand`)
   - Matrice des flux entre quartiers (`agg_borough_flows`)
4. Charger la table de dimension `dim_zones` et les agrégats dans PostgreSQL.
5. Valider les performances et vérifier l'intégrité des tables Gold via SQL.


---
## 1. Pourquoi Charger la Couche Gold dans PostgreSQL ?

| Critère | Couche Silver (Data Lake / MinIO) | Couche Gold (Data Warehouse / PostgreSQL) |
| :--- | :--- | :--- |
| **Type de données** | Trajets individuels au niveau ligne (~2.8M de lignes) | Données agrégées & tables dimensionnelles |
| **Usage type** | Data Science, Machine Learning, audits détaillés | Reporting BI, Tableaux de bord exécutifs, requêtes ad-hoc |
| **Outils consommateurs** | Apache Spark, DuckDB, Trino, scripts Python | Metabase, PowerBI, Tableau, requêtes SQL instantanées |
| **Temps de réponse** | 5 à 30 secondes (scan distribué) | Moins de 50 millisecondes (grâce aux index B-Tree) |


---
## 2. Initialisation de la Session PySpark et Connexion PostgreSQL


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from sqlalchemy import create_engine, text
import pandas as pd

minio_endpoint = os.getenv("MINIO_ENDPOINT", "http://minio:9000")
minio_access_key = os.getenv("MINIO_ACCESS_KEY", "admin")
minio_secret_key = os.getenv("MINIO_SECRET_KEY", "password123")

spark = (
    SparkSession.builder
    .appName("03_Chargement_Gold")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.hadoop.fs.s3a.endpoint", minio_endpoint)
    .config("spark.hadoop.fs.s3a.access.key", minio_access_key)
    .config("spark.hadoop.fs.s3a.secret.key", minio_secret_key)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

# Connexion SQLAlchemy à PostgreSQL
pg_host = os.getenv("POSTGRES_HOST", "postgres")
pg_port = os.getenv("POSTGRES_PORT", "5432")
pg_db = os.getenv("POSTGRES_DB", "gold_db")
pg_user = os.getenv("POSTGRES_USER", "postgres")
pg_pwd = os.getenv("POSTGRES_PASSWORD", "postgres123")

pg_engine = create_engine(f"postgresql+psycopg2://{pg_user}:{pg_pwd}@{pg_host}:{pg_port}/{pg_db}")
print("✅ PySpark et PostgreSQL prêts !")


---
## 3. Chargement de la Couche Silver depuis MinIO


In [ ]:
silver_taxi_path = "s3a://silver/nyc_taxi"
silver_zone_path = "s3a://bronze/reference/taxi_zone_lookup.csv"

df_silver = spark.read.parquet(silver_taxi_path)
df_zones_raw = spark.read.option("header", "true").option("inferSchema", "true").csv(silver_zone_path)

print(f"📥 Couche Silver chargée : {df_silver.count():,} trajets nettoyés disponibles pour l'agrégation.")


---
## 4. Construction et Insertion de la Table Dimensionnelle : `dim_zones`


In [ ]:
# Préparation du schéma dimensionnel pour PostgreSQL
df_dim_zones = (
    df_zones_raw
    .select(
        F.col("LocationID").alias("location_id"),
        F.col("Borough").alias("borough"),
        F.col("Zone").alias("zone_name"),
        F.col("service_zone")
    )
    .toPandas()
)

print(f"📍 Nombre de zones géographiques : {len(df_dim_zones)}")

# Insertion dans PostgreSQL
with pg_engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.dim_zones;"))
    df_dim_zones.to_sql("dim_zones", con=conn, schema="gold", if_exists="append", index=False)

print("✅ Table dimensionnelle 'gold.dim_zones' alimentée avec succès !")


---
## 5. Mart 1 : Métriques Quotidiennes (`agg_daily_taxi_metrics`)

Ce Data Mart répond aux questions de la direction financière et des opérations :
- Quel est le chiffre d'affaires quotidien par arrondissement ?
- Quelle est la distance moyenne et la durée moyenne d'une course chaque jour ?
- Quel est le taux de pourboire moyen ?


In [ ]:
df_daily_agg = (
    df_silver
    .groupBy("pickup_date", "pickup_borough", "VendorID")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.sum("fare_amount"), 2).alias("total_fare"),
        F.round(F.sum("tip_amount"), 2).alias("total_tips"),
        F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_min"),
        F.round(F.avg("tip_rate"), 4).alias("avg_tip_rate")
    )
    .withColumnRenamed("pickup_borough", "borough")
    .withColumnRenamed("VendorID", "vendor_id")
    .orderBy("pickup_date", "borough")
)

print("Aperçu du Data Mart Quotidien :")
df_daily_agg.show(5)

# Conversion en Pandas pour insertion optimisée dans Postgres
pdf_daily = df_daily_agg.toPandas()
print(f"📊 Volume de lignes agrégées : {len(pdf_daily)}")

with pg_engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.agg_daily_taxi_metrics;"))
    pdf_daily.to_sql("agg_daily_taxi_metrics", con=conn, schema="gold", if_exists="append", index=False)

print("✅ Table 'gold.agg_daily_taxi_metrics' alimentée dans PostgreSQL !")


---
## 6. Mart 2 : Analyse de la Demande Horaire & Vitesse (`agg_hourly_demand`)

Ce Data Mart permet d'analyser les heures de pointe (rush hours), la congestion du trafic et la vitesse moyenne à New York selon le moment de la semaine.


In [ ]:
df_hourly_agg = (
    df_silver
    .groupBy("pickup_hour", "day_of_week", "day_name")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min"),
        F.round(F.avg("speed_mph"), 2).alias("avg_speed_mph"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy("pickup_hour", "day_of_week")
)

print("Aperçu du Data Mart Horaire :")
df_hourly_agg.show(5)

pdf_hourly = df_hourly_agg.toPandas()

with pg_engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.agg_hourly_demand;"))
    pdf_hourly.to_sql("agg_hourly_demand", con=conn, schema="gold", if_exists="append", index=False)

print("✅ Table 'gold.agg_hourly_demand' alimentée dans PostgreSQL !")


---
## 7. Mart 3 : Matrice des Flux entre Arrondissements (`agg_borough_flows`)

Ce Data Mart met en lumière les trajets les plus fréquents et les plus lucratifs (ex: Manhattan ➔ JFK / Queens).


In [ ]:
df_flows_agg = (
    df_silver
    .groupBy("pickup_borough", "dropoff_borough")
    .agg(
        F.count("*").alias("total_trips"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
        F.round(F.avg("trip_duration_minutes"), 2).alias("avg_duration_min")
    )
    .orderBy(F.desc("total_trips"))
)

print("Top 5 des flux entre arrondissements :")
df_flows_agg.show(5)

pdf_flows = df_flows_agg.toPandas()

with pg_engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE gold.agg_borough_flows;"))
    pdf_flows.to_sql("agg_borough_flows", con=conn, schema="gold", if_exists="append", index=False)

print("✅ Table 'gold.agg_borough_flows' alimentée dans PostgreSQL !")


---
## 8. Validation dans PostgreSQL avec Requêtes SQL

Exécutons une requête SQL directement sur la vue métier `gold.v_top_performing_boroughs` créée dans PostgreSQL lors de l'initialisation du schéma.


In [ ]:
with pg_engine.connect() as conn:
    query = text("""
        SELECT 
            borough,
            total_rides,
            gross_revenue,
            overall_avg_distance,
            avg_tip_percentage
        FROM gold.v_top_performing_boroughs;
    """)
    df_result = pd.read_sql(query, conn)

print("🏆 Résultat de la vue SQL métier dans PostgreSQL :")
print(df_result.to_string(index=False))


---
## 🏁 Conclusion du Notebook 03
- Les données ont franchi la dernière étape du pipeline : **la Couche Gold**.
- Les Data Marts sont structurés, indexés et stockés dans PostgreSQL.
- Les analystes métier et les décideurs peuvent désormais interroger ces tables avec des temps de réponse instantanés.

👉 Passez au notebook **`04_exploration_et_metabase.ipynb`** pour explorer les visualisations BI avec **Metabase** !
